In [ ]:
from pathlib import Path
ROOT = Path().resolve()
while not (ROOT / "DATA").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
DATA = ROOT / "DATA"
print("CLEAN root:", ROOT)


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:

import os
import sys
sys.path.insert(0, "..")  # pour importer src/

os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"

import torch
from sentence_transformers import SentenceTransformer

from src.config import Config
from src.data_loading import load_questions, load_article_ids, filter_questions_by_articles, sample_df
from src.graph_utils import ensure_graph_artifacts
from src.retrieval_rag import ensure_rag_index
from src.llm import load_llm
from src.evaluation import run_all_modes
from src.inspection import inspect_examples
from src.io_utils import save_results
from src.filter_components import filter_graph_json_to_giant_component


In [ ]:
#test cuda :
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

In [ ]:
# import json
# import networkx as nx


# filter_graph_json_to_giant_component(
#         input_json_path="str(DATA / "SUBSETS_ES" / "GRAPHS" / "gastronomia" / "gastronomia_clustered_graph.json")",
#         output_json_path="# (no equivalent in CLEAN)",
#     )


In [ ]:
cfg = Config(
    # Décommente / modifie ce que tu veux changer
    CATEGORY="global",
    graph_dir=str(DATA / "SUBSETS_ES" / "GRAPHS" / "global"),
    n_samples=None,
    modes_to_run=["zero_shot","graph_rag_pcst","rag","kaping"],
    force_rebuild_graph_artifacts=False,
    force_rebuild_rag_index=False,
    device="cuda:0"
)
print(cfg)


In [ ]:
# import pandas as pd
# #créer les articles globaux, en mergeant les articles de toutes les catégories:
# CATEGORIES = [
#     "artesania", 
#     "literatura", 
#     "folclore",
#     "gastronomia",
#     "danza", "pintura", "musica", "cine"
# ]
# df_global=pd.DataFrame()

# #on rajoute une colonne catégorie à chaque df_category pour pouvoir filtrer plus tard
# for category in CATEGORIES:
#     df_category=pd.read_csv(str(DATA / "SUBSETS_ES" / "ARTICLES_SUBSETS_ES" / f"{category}_articles_es_disjoint.csv"))
#     df_category["category"]=category
#     df_global=pd.concat([df_global, df_category], ignore_index=True)

# #afficher les stats de df_global

# print("Nombre total d'articles globaux:", len(df_global))
# print("nombre d'articles par catégorie:")
# print(df_global["category"].value_counts())
# df_global.to_csv(str(DATA / "SUBSETS_ES" / "ARTICLES_SUBSETS_ES" / "global_articles_es_disjoint.csv"), index=False)

In [ ]:
df_questions = load_questions(cfg.questions_csv)

if cfg.filter_by_articles:
    article_ids = load_article_ids(cfg.articles_csv)
    df_questions = filter_questions_by_articles(df_questions, article_ids)

# ⚠️ Cellule à modifier si tu veux changer le df évalué
df_eval = sample_df(df_questions, cfg.n_samples, cfg.random_seed)
print(f"📊 df_eval : {len(df_eval)} questions")


In [ ]:
print(cfg.graph_dir)
print(cfg.graph_json_name)

In [ ]:
# # GRAPHE JSON :
# import pickle 
# print("ouverture du graphe depuis pickle :")
# def load_graph_from_pickle(pickle_path: str):
#     """Charge un graphe depuis un fichier pickle."""
#     with open(pickle_path, "rb") as f:
#         return pickle.load(f)

# clustered_graph = load_graph_from_pickle(cfg.graph_dir + "/global_clustered_graph.pkl")   

# import json
# from pathlib import Path

# # Convertir le graphe en dict sérialisable
# def graph_to_dict(graph):
#     return {
#         "entities": sorted(list(graph.entities)) if graph.entities else [],
#         "edges": sorted(list(graph.edges)) if graph.edges else [],
#         "relations": [list(r) for r in sorted(graph.relations)] if graph.relations else [],
#         "entity_clusters": {
#             k: sorted(list(v)) for k, v in graph.entity_clusters.items()
#         } if graph.entity_clusters else {},
#         "edge_clusters": {
#             k: sorted(list(v)) for k, v in graph.edge_clusters.items()
#         } if graph.edge_clusters else {},
#     }

# # Sauvegarde
# output_path = Path(cfg.graph_dir) / cfg.graph_json_name
# with open(output_path, "w", encoding="utf-8") as f:
#     json.dump(graph_to_dict(clustered_graph), f, ensure_ascii=False, indent=2)

# print(f"✅ Graphe sauvegardé en JSON : {output_path}")


In [ ]:
graph_embedder = SentenceTransformer(cfg.graph_embedder_id, device=cfg.device, trust_remote_code=True)
nodes_df, edges_df, node2id, graph = ensure_graph_artifacts(cfg, graph_embedder)


In [ ]:
rag_embedder = SentenceTransformer(cfg.rag_embedder_id, device=cfg.device, trust_remote_code=True)
rag_index, rag_chunks_df = ensure_rag_index(cfg)
# rag_embedder= None
# rag_index,rag_chunks_df = None, None



In [ ]:
llm, tokenizer = load_llm(cfg.llm_id, device=cfg.device)


In [ ]:
# # 1) cache PCST (1 fois)
# pcst_cache = build_pcst_cache(
#     df_questions, cfg, graph, nodes_df, edges_df,
#     graph_embedder, gcfg.pcst_cache_path,
# )
# pcst_cache_path = cfg.graph_dir + "/pcst_cache_for_gnn"
# os.makedirs(pcst_cache_path, exist_ok=True)
# #save as pickle
# import pickle
# with open(pcst_cache_path + "/pcst_cache.pkl", "wb") as f:
#     pickle.dump(pcst_cache, f)

In [ ]:
#OUVRE LE PICKLE DU CACHE PCST : 
import pickle
cache_path = cfg.graph_dir + "/pcst_cache_for_gnn/pcst_cache.pkl"
with open(cache_path, "rb") as f:
    pcst_cache = pickle.load(f)

In [ ]:
len(pcst_cache)

In [ ]:
import copy
import torch
from src.gnn_utils.gnn_config import GNNConfig
from src.gnn_utils.g_retriever_model import GRetriever
from src.gnn_utils.train import train_g_retriever, make_splits
from src.gnn_utils.dataset import GRetrieverMCQDataset

# --- 1) Un SEUL split partagé par les 3 modes ---
base_cfg = GNNConfig(train_gen_mode="gen_letter")                      # ta config de base
dataset_tmp = GRetrieverMCQDataset(df_questions, pcst_cache, graph)


fixed_splits = make_splits(len(dataset_tmp), base_cfg)
print("Tailles -> train/val/test :", [len(s) for s in fixed_splits])

# --- 2) Définition des 3 ablations ---
ablations = {
    "text_only":  dict(use_graph_token=False, use_text_graph=True),
    # "graph_only": dict(use_graph_token=True,  use_text_graph=False),
    # "both":       dict(use_graph_token=True,  use_text_graph=True),
}

results = {}

for name, flags in ablations.items():
    print(f"\n{'='*60}\n>>> ABLATION : {name}  ({flags})\n{'='*60}")

    # config dédiée (copie pour ne pas polluer base_cfg)
    cfg = copy.deepcopy(base_cfg)
    cfg.use_graph_token = flags["use_graph_token"]
    cfg.use_text_graph  = flags["use_text_graph"]
    cfg.ablation_name   = name          # <-- dossier de sortie dedie
    # checkpoint distinct par mode pour ne pas écraser
    cfg.ckpt_path = base_cfg.ckpt_path.replace(".pt", f"_{name}.pt")

    # modèle neuf à chaque fois (GNN + projector réinitialisés)
    model = GRetriever(llm, tokenizer, cfg)
    model.llm.gradient_checkpointing_enable()
    model.llm.config.use_cache = False

    res = train_g_retriever(
        model, df_questions, pcst_cache, graph, cfg,
        splits=fixed_splits,           # <-- même split pour tous
    )
    results[name] = res
    # libère la mémoire GPU entre deux modes
    del model
    torch.cuda.empty_cache()
# --- 3) Récap ---
print(f"\n\n{'='*70}\n📊 RÉCAPITULATIF (même split, mode={base_cfg.train_gen_mode})\n{'='*70}")
header = f"{'mode':<12} {'val_acc':>10} {'test_acc':>10}"
print(header)
for name, res in results.items():
    print(f"{name:<12} "
          f"{res['best_val_acc']:>10.4f} "
          f"{res['test_acc']:>10.4f}")


## Grid Search : `num_graph_tokens` × `lr` (ablation `both`, `gen_letter`)

On fixe `use_graph_token=True`, `use_text_graph=True` (mode `both`) et on fait varier :
- `num_graph_tokens` ∈ [1, 2, 4] — nombre de soft tokens injectés
- `lr` ∈ [5e-5, 1e-4] — learning rate

Même `fixed_splits` que l'ablation pour comparaison équitable.  
Chaque config a son propre dossier de résultats (details, summary, best.pt, …) géré par `train_g_retriever`.


In [ ]:
# === GRID SEARCH : num_graph_tokens × lr (ablation "both", gen_letter) ===
import copy, torch, pandas as pd
from pathlib import Path
from src.gnn_utils.gnn_config import GNNConfig
from src.gnn_utils.g_retriever_model import GRetriever
from src.gnn_utils.train import train_g_retriever, make_splits

# --- Config de base (both + gen_letter) ---
base_cfg = GNNConfig(
    train_gen_mode="gen_full",
    use_graph_token=True,
    use_text_graph=True,
    category="global",
)



dataset_tmp = GRetrieverMCQDataset(df_questions, pcst_cache, graph)
fixed_splits = make_splits(len(dataset_tmp), base_cfg)
print("Tailles -> train/val/test :", [len(s) for s in fixed_splits])


# --- Grille 6 configs ---
grid = [
    {"num_graph_tokens": ngt, "lr": lr}
    for ngt in [1,
    for lr  in [5e-5, 1e-4]
]

grid_results = []

for cfg_idx, hp in enumerate(grid, 1):
    ngt, lr = hp["num_graph_tokens"], hp["lr"]
    label = f"ngt={ngt}_lr={lr}"
    print(f"\n{'='*70}\n  GRID {cfg_idx}/{len(grid)} : {label}\n{'='*70}")

    cfg = copy.deepcopy(base_cfg)
    cfg.num_graph_tokens = ngt
    cfg.lr = lr
    cfg.ablation_name = f"both_ngt{ngt}_lr{lr}"

    model = GRetriever(llm, tokenizer, cfg)
    model.llm.gradient_checkpointing_enable()
    model.llm.config.use_cache = False

    res = train_g_retriever(
        model, df_questions, pcst_cache, graph, cfg,
        splits=fixed_splits,
    )
    grid_results.append({
        "num_graph_tokens": ngt,
        "lr": lr,
        "best_val_acc": res["best_val_acc"],
        "test_acc": res["test_acc"],
        "best_epoch": res.get("best_epoch", 0),
        "run_dir": res["run_dir"],
    })

    del model
    torch.cuda.empty_cache()

# --- Récap ---
grid_df = pd.DataFrame(grid_results).sort_values("test_acc", ascending=False)
print(f"\n{'='*70}\n  GRID SEARCH RÉCAP (both, gen_letter)\n{'='*70}")
print(grid_df.to_string(index=False))

# --- Sauvegarde CSV global ---
llm_slang = llm.name_or_path.split("/")[-1].replace("-", "_")
csv_path = Path(base_cfg.results_root) / llm_slang / "grid_search_results.csv"
csv_path.parent.mkdir(parents=True, exist_ok=True)
grid_df.to_csv(csv_path, index=False)
print(f"\nSaved: {csv_path}")

## K FOLD : 

In [ ]:
# ============================================================
# K-FOLD CROSS-VALIDATION pour G-Retriever
# Stratifié par catégorie, mêmes folds pour toutes les ablations
# ============================================================
import copy
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, train_test_split
from scipy import stats

from src.gnn_utils.gnn_config import GNNConfig
from src.gnn_utils.g_retriever_model import GRetriever
from src.gnn_utils.train import train_g_retriever
from src.gnn_utils.dataset import GRetrieverMCQDataset

# ------------------------------------------------------------
# 0) Paramètres de la CV
# ------------------------------------------------------------
K_FOLDS = 5
SEED    = 1          # aligné sur cfg.random_seed / cfg.seed

# ------------------------------------------------------------
# 1) Dataset (une seule fois) + labels de stratification
# ------------------------------------------------------------
dataset = GRetrieverMCQDataset(df_questions, pcst_cache, graph)
n_total = len(dataset)

# Stratification par catégorie (tu as 8 catégories) — idéal pour ton dataset global
if "category" in df_questions.columns:
    strat_labels = df_questions["category"].to_numpy()
    print(f"Stratification par 'category' : {pd.Series(strat_labels).value_counts().to_dict()}")
else:
    strat_labels = df_questions["correct_letter"].to_numpy()  # ADAPTER si autre nom
    print("Stratification par 'correct_letter' (fallback)")

assert len(strat_labels) == n_total, (
    f"Désalignement labels ({len(strat_labels)}) vs dataset ({n_total}). "
    "Vérifie que GRetrieverMCQDataset suit l'ordre de df_questions."
)

# ------------------------------------------------------------
# 2) Générer les folds (MÊMES folds pour toutes les ablations)
#    On respecte tes ratios : test = 1/K, val = cfg.val_ratio du reste
# ------------------------------------------------------------
def make_kfold_splits(n, labels, k, val_ratio, seed):
    """Liste de k tuples (train_idx, val_idx, test_idx), stratifiés.
    Le val est extrait du train de chaque fold (stratifié aussi)."""
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
    all_idx = np.arange(n)
    folds = []
    for trainval_idx, test_idx in skf.split(all_idx, labels):
        # proportion de val relative au (train+val), pour rester ~= cfg.val_ratio global
        rel_val = val_ratio / (1.0 - 1.0 / k)
        train_idx, val_idx = train_test_split(
            trainval_idx,
            test_size=rel_val,
            stratify=labels[trainval_idx],
            random_state=seed,
        )
        folds.append((
            np.sort(train_idx).tolist(),
            np.sort(val_idx).tolist(),
            np.sort(test_idx).tolist(),
        ))
    return folds

# on lit val_ratio depuis une GNNConfig de référence
_ref_cfg = GNNConfig(category="global",device="cuda:1",batch_size=1)
kfold_splits = make_kfold_splits(
    n_total, strat_labels, k=K_FOLDS, val_ratio=_ref_cfg.val_ratio, seed=SEED
)

print(f"\nK-Fold ({K_FOLDS} folds) — tailles [train/val/test] :")
for i, (tr, va, te) in enumerate(kfold_splits):
    print(f"  fold {i}: {len(tr)} / {len(va)} / {len(te)}")

# ------------------------------------------------------------
# 3) Config de base — HYPERPARAMÈTRES FIGÉS (issus de ta grid search)
#    ⚠️ NE PAS refaire de grid search dans la CV = éviter le data leakage
# ------------------------------------------------------------
base_cfg = GNNConfig(
    train_gen_mode="gen_full",
    use_graph_token=True,
    use_text_graph=True,
    category="global",
    device="cuda:0",
    batch_size=1
)

# ------------------------------------------------------------
# 4) Ablations (mêmes folds pour toutes)
# ------------------------------------------------------------
ablations = {
    #  "text_only":  dict(use_graph_token=False, use_text_graph=True),
    # "graph_only": dict(use_graph_token=True,  use_text_graph=False),
    "both":       dict(use_graph_token=True,  use_text_graph=True),
}
# ------------------------------------------------------------
# 5) Boucle CV avec SAVE INCRÉMENTAL + REPRISE + tolérance aux pannes
# ------------------------------------------------------------
import traceback
from datetime import datetime

llm_slang = llm.name_or_path.split("/")[-1].replace("-", "_")
out_dir = Path(base_cfg.results_root) / llm_slang / base_cfg.category / "kfold_cv"
out_dir.mkdir(parents=True, exist_ok=True)

progress_csv = out_dir / "kfold_details.csv"   # mis à jour après CHAQUE run
splits_path  = out_dir / "kfold_splits.npy"

# sauvegarde des folds tout de suite (repro + reprise)
np.save(splits_path, np.array(kfold_splits, dtype=object), allow_pickle=True)

# --- REPRISE : recharger ce qui est déjà fait ---
if progress_csv.exists():
    cv_df = pd.read_csv(progress_csv)
    # On ne considère "fait" QUE les runs dont status == "ok".
    # Tout run avec un status différent de "ok" (FAILED, etc.) sera repris.
    if "status" in cv_df.columns:
        done_mask = cv_df["status"] == "ok"
    else:
        done_mask = pd.Series(False, index=cv_df.index)

    done = set(zip(cv_df.loc[done_mask, "fold"], cv_df.loc[done_mask, "ablation"]))

    # runs à reprendre = présents dans le CSV mais status != "ok"
    to_retry = set(zip(cv_df.loc[~done_mask, "fold"], cv_df.loc[~done_mask, "ablation"]))
    to_retry -= done  # au cas où un couple aurait à la fois un ok et un failed

    print(f"♻️  Reprise : {len(done)} run(s) 'ok' -> {sorted(done)}")
    if to_retry:
        print(f"🔁 À reprendre (status != 'ok') : {sorted(to_retry)}")
else:
    cv_df = pd.DataFrame()
    done = set()

def append_row(row: dict):
    """Ajoute/remplace une ligne (fold, ablation) et réécrit le CSV IMMÉDIATEMENT (crash-safe)."""
    global cv_df
    # supprime une éventuelle ancienne ligne pour ce (fold, ablation)
    if not cv_df.empty and {"fold", "ablation"}.issubset(cv_df.columns):
        cv_df = cv_df[~(
            (cv_df["fold"] == row["fold"]) &
            (cv_df["ablation"] == row["ablation"])
        )].copy()

    cv_df = pd.concat([cv_df, pd.DataFrame([row])], ignore_index=True)
    # écriture atomique : tmp puis rename (évite un CSV corrompu si crash pendant l'écriture)
    tmp = progress_csv.with_suffix(".csv.tmp")
    cv_df.to_csv(tmp, index=False)
    tmp.replace(progress_csv)

failures = []

for fold_i, split in enumerate(kfold_splits):
    print(f"\n{'#'*70}\n# FOLD {fold_i+1}/{K_FOLDS}\n{'#'*70}")

    for name, flags in ablations.items():

        # --- SKIP si déjà fait (reprise) ---
        if (fold_i, name) in done:
            print(f"⏭  skip fold={fold_i} ablation={name} (déjà fait)")
            continue

        print(f"\n{'='*60}\n>>> fold={fold_i}  ablation={name}  {flags}\n{'='*60}")

        try:
            cfg = copy.deepcopy(base_cfg)
            cfg.use_graph_token = flags["use_graph_token"]
            cfg.use_text_graph  = flags["use_text_graph"]
            cfg.ablation_name   = f"cv_fold{fold_i}_{name}"
            cfg.ckpt_path = base_cfg.ckpt_path.replace(".pt", f"_fold{fold_i}_{name}.pt")

            model = GRetriever(llm, tokenizer, cfg)
            model.llm.gradient_checkpointing_enable()
            model.llm.config.use_cache = False

            res = train_g_retriever(
                model, df_questions, pcst_cache, graph, cfg,
                splits=split,
            )

            append_row({
                "fold": fold_i,
                "ablation": name,
                "val_acc":   res["best_val_acc"],
                "test_acc":  res["test_acc"],
                "best_epoch": res.get("best_epoch", 0),
                "run_dir":   res.get("run_dir", None),
                "status":    "ok",
                "timestamp": datetime.now().isoformat(timespec="seconds"),
            })
            print(f"💾 sauvegardé -> {progress_csv.name} "
                  f"({len(cv_df)} run(s) au total)")

            del model
            torch.cuda.empty_cache()

        except Exception as e:
            # --- un run plante : on log, on libère, on CONTINUE ---
            tb = traceback.format_exc()
            print(f"❌ ÉCHEC fold={fold_i} ablation={name} : {e}")
            print(tb)
            failures.append((fold_i, name, str(e)))

            # trace l'échec dans le CSV pour ne pas le reprendre en boucle infinie
            append_row({
                "fold": fold_i,
                "ablation": name,
                "val_acc": None, "test_acc": None, "best_epoch": None,
                "run_dir": None, "status": f"FAILED: {e}",
                "timestamp": datetime.now().isoformat(timespec="seconds"),
            })

            # log détaillé dans un fichier séparé
            with open(out_dir / "failures.log", "a") as f:
                f.write(f"\n{'='*60}\nfold={fold_i} ablation={name} "
                        f"@ {datetime.now().isoformat()}\n{tb}\n")

            # nettoyage mémoire pour ne pas contaminer le run suivant
            try:
                del model
            except NameError:
                pass
            torch.cuda.empty_cache()

# ------------------------------------------------------------
# 6) Agrégation FINALE (ignore les runs échoués)
# ------------------------------------------------------------
ok_df = cv_df[cv_df["status"] == "ok"].copy()
ok_df["test_acc"] = pd.to_numeric(ok_df["test_acc"])

if failures:
    print(f"\n⚠️  {len(failures)} run(s) échoué(s) : {failures}")
    print("   -> relance simplement la cellule : ils seront repris, les 'ok' seront skippés.")

summary = (
    ok_df.groupby("ablation")["test_acc"]
    .agg(test_acc_mean="mean", test_acc_std="std",
         test_acc_min="min", test_acc_max="max", n="count")
    .sort_values("test_acc_mean", ascending=False)
)
print(f"\n{'='*70}\n📊 K-FOLD RÉCAP ({K_FOLDS} folds)\n{'='*70}")
print(summary.to_string())
summary.to_csv(out_dir / "kfold_summary.csv")

# pivot uniquement sur les folds COMPLETS (toutes ablations ok)
complete_folds = (
    ok_df.groupby("fold")["ablation"].nunique()
    .loc[lambda s: s == len(ablations)].index
)
if len(complete_folds) >= 2:
    print("\nDétail par fold (folds complets, test_acc) :")
    pivot = (ok_df[ok_df["fold"].isin(complete_folds)]
             .pivot(index="fold", columns="ablation", values="test_acc"))
    print(pivot.to_string())

    # tests appariés Wilcoxon sur folds complets
    print(f"\n🔬 Tests appariés (Wilcoxon, {len(complete_folds)} folds complets)")
    abl_names = list(pivot.columns)
    for i in range(len(abl_names)):
        for j in range(i + 1, len(abl_names)):
            a, b = abl_names[i], abl_names[j]
            try:
                w, p = stats.wilcoxon(pivot[a], pivot[b])
                print(f"  {a} vs {b}: W={w:.2f} p={p:.4f} "
                      f"(Δmean={pivot[a].mean()-pivot[b].mean():+.4f})")
            except ValueError as e:
                print(f"  {a} vs {b}: N/A ({e})")
else:
    print("\n⚠️  Pas assez de folds complets pour le test apparié.")

print(f"\n✅ Tout sauvegardé dans : {out_dir}")

In [ ]:
# # ============================================================
# # CELL D'ANALYSE : inspecter manuellement les générations
# # Compare raw_output vs parsed_letter pour comprendre les erreurs de parsing
# # ============================================================

# import copy, torch, re
# from src.gnn_utils.gnn_config import GNNConfig
# from src.gnn_utils.g_retriever_model import GRetriever, parse_letter
# from src.gnn_utils.train import make_splits
# from src.gnn_utils.dataset import GRetrieverMCQDataset, mcq_collate
# from torch.utils.data import DataLoader, Subset

# # --- Config : choisir le mode à analyser ---
# ANALYZE_MODE = "both"   # "both", "graph_only", ou "text_only"
# GEN_MODE     = "letter"  # "letter" ou "full"

# flags = {
#     "text_only":  dict(use_graph_token=False, use_text_graph=True),
#     "graph_only": dict(use_graph_token=True,  use_text_graph=False),
#     "both":       dict(use_graph_token=True,  use_text_graph=True),
# }[ANALYZE_MODE]

# cfg = GNNConfig()
# cfg.use_graph_token = flags["use_graph_token"]
# cfg.use_text_graph  = flags["use_text_graph"]
# cfg.train_gen_mode  = f"gen_{GEN_MODE}"
# cfg.ckpt_path = cfg.ckpt_path.replace(".pt", f"_{ANALYZE_MODE}.pt")

# dataset = GRetrieverMCQDataset(df_questions, pcst_cache, graph)
# tr_idx, va_idx, te_idx = make_splits(len(dataset), cfg)
# test_loader = DataLoader(Subset(dataset, te_idx), batch_size=1,
#                          shuffle=False, collate_fn=mcq_collate)

# # --- Reconstruire le modèle et charger le best checkpoint ---
# model = GRetriever(llm, tokenizer, cfg)
# model.llm.gradient_checkpointing_enable()
# model.llm.config.use_cache = False
# ckpt = torch.load(cfg.ckpt_path, map_location=cfg.device)
# model.gnn.load_state_dict(ckpt["gnn"])
# model.projector.load_state_dict(ckpt["projector"])
# model.gnn.to(cfg.device)
# model.projector.to(cfg.device)
# model.eval()

# # --- Collecte des raw generations ---
# rows = []
# for i, batch in enumerate(test_loader):
#     for k in ("x", "edge_index", "edge_attr", "batch"):
#         batch[k] = batch[k].to(cfg.device)

#     # génération greedy
#     with torch.no_grad():
#         preds, raws = model.generate_answers(batch, gen_mode=GEN_MODE, debug=False)

#     # pour récupérer le raw text, on refait la gen manuellement
#     with torch.no_grad():
#         llm_dtype = next(model.llm.parameters()).dtype
#         if cfg.use_graph_token:
#             graph_tokens = model._graph_tokens(batch).to(llm_dtype)
#         else:
#             graph_tokens = None

#         prompt = model._build_prompt(
#             batch["desc"][0], batch["question"][0],
#             batch["options"][0], mode=f"gen_{GEN_MODE}",
#         )
#         p_ids = torch.tensor(
#             model.tokenizer(prompt, add_special_tokens=False,
#                             truncation=True, max_length=cfg.max_txt_len).input_ids
#         ).to(cfg.device)
#         p_emb = model._embed_text(p_ids)

#         if cfg.use_graph_token and graph_tokens is not None:
#             full_emb = torch.cat([graph_tokens[0], p_emb], dim=0).unsqueeze(0)
#         else:
#             full_emb = p_emb.unsqueeze(0)
#         full_emb = full_emb.to(llm_dtype)
#         attn = torch.ones(full_emb.size(1), device=cfg.device, dtype=torch.long).unsqueeze(0)

#         mnt = 5 if GEN_MODE == "letter" else 64
#         gen = model.llm.generate(
#             inputs_embeds=full_emb, attention_mask=attn,
#             max_new_tokens=mnt, do_sample=False,
#             pad_token_id=model.tokenizer.eos_token_id,
#         )
#         raw = model.tokenizer.decode(gen[0], skip_special_tokens=True)

#     gold = batch["correct_letter"][0]
#     pred = preds[0] if preds else None
#     parsed = parse_letter(raw) if GEN_MODE == "letter" else None

#     rows.append({
#         "idx": te_idx[i],
#         "question": batch["question"][0][:80],
#         "gold": gold,
#         "pred": pred,
#         "parsed": parsed,
#         "raw": repr(raw),
#         "raw_len": len(raw),
#         "prompt_len": len(prompt),
#         "correct": pred == gold,
#         "unparsed": pred is None,
#     })

# # --- Affichage : d'abord les unparsed, puis les wrong ---
# import pandas as pd
# df_dbg = pd.DataFrame(rows)

# n_unparsed = df_dbg["unparsed"].sum()
# n_wrong = (~df_dbg["unparsed"] & ~df_dbg["correct"]).sum()
# n_ok = df_dbg["correct"].sum()
# print(f"\n{'='*70}")
# print(f"ANALYSE : mode={ANALYZE_MODE}  gen={GEN_MODE}")
# print(f"  Total: {len(df_dbg)}  |  Correct: {n_ok}  |  Wrong: {n_wrong}  |  Unparsed: {n_unparsed}")
# print(f"{'='*70}\n")

# if n_unparsed > 0:
#     print(f"--- ❌ UNPARSED ({n_unparsed}) ---")
#     for _, r in df_dbg[df_dbg["unparsed"]].iterrows():
#         print(f"  idx={r['idx']}  gold={r['gold']}  raw={r['raw']}  raw_len={r['raw_len']}  prompt_len={r['prompt_len']}")
#         print(f"    Q: {r['question']}")
#         print()

# if n_wrong > 0:
#     print(f"--- ⚠️ WRONG (parsed mais faux) ({n_wrong}) ---")
#     for _, r in df_dbg[~df_dbg["unparsed"] & ~df_dbg["correct"]].iterrows():
#         print(f"  idx={r['idx']}  gold={r['gold']}  pred={r['pred']}  raw={r['raw']}  prompt_len={r['prompt_len']}")
#         print(f"    Q: {r['question']}")
#         print()

# # --- Stats sur la longueur des prompts (unparsed vs parsed) ---
# print(f"--- 📏 Prompt length: unparsed vs parsed ---")
# if n_unparsed > 0:
#     print(f"  unparsed: mean={df_dbg[df_dbg['unparsed']]['prompt_len'].mean():.0f}  "
#           f"min={df_dbg[df_dbg['unparsed']]['prompt_len'].min()}  "
#           f"max={df_dbg[df_dbg['unparsed']]['prompt_len'].max()}")
# parsed_mask = ~df_dbg["unparsed"]
# print(f"  parsed:   mean={df_dbg[parsed_mask]['prompt_len'].mean():.0f}  "
#       f"min={df_dbg[parsed_mask]['prompt_len'].min()}  "
#       f"max={df_dbg[parsed_mask]['prompt_len'].max()}")

# # --- Inspecter un exemple en détail (prompt complet) ---
# SHOW_IDX = None  # mettre un idx pour voir le prompt complet, sinon = premier unparsed
# if SHOW_IDX is None and n_unparsed > 0:
#     SHOW_IDX = df_dbg[df_dbg["unparsed"]].iloc[0]["idx"]
# if SHOW_IDX is not None:
#     row = df_questions.iloc[SHOW_IDX]
#     c = pcst_cache[SHOW_IDX]
#     prompt_full = model._build_prompt(
#         c["desc"], row["question"],
#         {L: str(row[f"option {L}"]) for L in ["A","B","C","D"]},
#         mode=f"gen_{GEN_MODE}",
#     )
#     print(f"\n{'='*70}")
#     print(f"EXEMPLE DÉTAILLÉ (idx={SHOW_IDX})")
#     print(f"{'='*70}")
#     print(f"PROMPT ({len(prompt_full)} chars):")
#     print(prompt_full)
#     print(f"\nRAW GENERATION: {repr(rows[SHOW_IDX if SHOW_IDX < len(rows) else 0]['raw'])}")
#     print(f"PARSED: {rows[SHOW_IDX if SHOW_IDX < len(rows) else 0]['parsed']}")
    print(f"GOLD:   {rows[SHOW_IDX if SHOW_IDX < len(rows) else 0]['gold']}")